# 02 — Debugging Strategies

Reading error messages is the first step. But many bugs don't raise errors at all —
they silently compute the wrong answer. This notebook covers practical strategies
for finding and fixing both crashing bugs and silent logic errors.

## Learning Objectives

- Use print debugging to trace program state
- Write `assert` statements to add defensive checks
- Read function call stacks to understand execution flow
- Recognize common patterns: off-by-one errors, mutation bugs, indentation issues
- Develop the debugging mindset: expected vs. actual
- Know what `pdb` and `breakpoint()` are for

## Setup

In [ ]:
import sys
sys.path.insert(0, "../../")
from src.checks import check_equal, check_type, check_contains

print("Setup complete.")

---
## The Debugging Mindset

Before reaching for any tool, ask two questions:

1. **What did I expect to happen?** Be specific: "I expected `scores` to be a list of 5 floats between 0 and 1."
2. **What actually happened?** Print it out. Don't guess.

Most bugs are found by making the gap between those two answers visible.

This is exactly the same approach you'd use debugging Node.js — the tools are just slightly different.

---
## 1. Print Debugging

Print debugging is the most universally used debugging technique — and it works.
The key is **strategic placement**: print at the boundaries of your understanding.

In [ ]:
# Example: a function to normalize scores
def normalize_scores(scores):
    """Scale scores to the range [0, 1] based on min and max."""
    min_score = min(scores)
    max_score = max(scores)
    
    print(f"[DEBUG] Input scores: {scores}")  # What came in?
    print(f"[DEBUG] min={min_score}, max={max_score}")  # Intermediate state

    normalized = []
    for i, s in enumerate(scores):
        n = (s - min_score) / (max_score - min_score)
        print(f"[DEBUG] scores[{i}] = {s} -> normalized = {n:.4f}")  # Per-iteration
        normalized.append(n)
    
    print(f"[DEBUG] Output: {normalized}")  # What went out?
    return normalized

raw_scores = [0.6, 0.8, 0.55, 0.9, 0.7]
result = normalize_scores(raw_scores)
print(f"Result: {result}")

In [ ]:
# Good print debugging habits:

# 1. Label your prints so you know where they came from
x = [1, 2, 3]
print(f"[normalize_scores] input: {x}")  # Better than just print(x)

# 2. Print the TYPE as well as the value — type bugs are common
value = "42"  # Looks like a number but it's a string!
print(f"value = {value!r}, type = {type(value).__name__}")
# !r uses repr(), which shows string quotes if they exist — makes it obvious if it's a str

# 3. Print length for collections
items = ["a", "b", "c"]
print(f"items has {len(items)} elements: {items}")

# 4. Use a separator to make output readable when there are many prints
print("-" * 40)
print("Starting new section")
print("-" * 40)

---
## 2. Assert Statements

`assert` lets you declare what you believe to be true at a given point in the code.
If the assertion is false, Python raises an `AssertionError` immediately — stopping execution
right at the point of the bug rather than letting bad data propagate silently.

Think of asserts as executable documentation of your assumptions.

In [ ]:
# Basic assert syntax:
# assert <condition>, "message shown if condition is False"

scores = [0.8, 0.6, 0.9]

# Assert the input is what we expect
assert isinstance(scores, list), f"Expected a list, got {type(scores)}"
assert len(scores) > 0, "scores list is empty"
assert all(0 <= s <= 1 for s in scores), f"Scores must be between 0 and 1, got: {scores}"

print("All assertions passed — inputs look good!")

# What happens when an assertion fails:
bad_scores = [0.5, 1.5, 0.8]  # 1.5 is out of range!
try:
    assert all(0 <= s <= 1 for s in bad_scores), f"Out-of-range score found: {bad_scores}"
except AssertionError as e:
    print(f"Caught assertion failure: {e}")

In [ ]:
# Pattern: add assertions at function boundaries

def compute_accuracy(results):
    """Compute the fraction of correct results."""
    # Assert preconditions (what the function requires)
    assert isinstance(results, list), "results must be a list"
    assert len(results) > 0, "results list is empty — cannot compute accuracy"
    assert all("correct" in r for r in results), "Each result must have a 'correct' field"
    
    correct = sum(1 for r in results if r["correct"] is True)
    accuracy = correct / len(results)
    
    # Assert postconditions (what the function guarantees)
    assert 0 <= accuracy <= 1, f"Accuracy out of range: {accuracy}"
    
    return accuracy

test_results = [
    {"id": 1, "correct": True},
    {"id": 2, "correct": False},
    {"id": 3, "correct": True},
]

acc = compute_accuracy(test_results)
print(f"Accuracy: {acc:.2%}")  # 66.67%

---
## 3. Reading Function Call Stacks

When you see a multi-level traceback, Python is showing you the exact chain of calls.
Understanding which function called which helps you pinpoint where the bad data entered the system.

In [ ]:
import traceback

# A chain of functions where the error happens at the bottom
def parse_score(raw_value):
    return float(raw_value)  # Will fail if raw_value is not numeric

def process_record(record):
    score = parse_score(record["score"])  # Calls parse_score
    return {"id": record["id"], "score": score}

def process_all(records):
    return [process_record(r) for r in records]  # Calls process_record for each

bad_data = [
    {"id": "r1", "score": "0.85"},
    {"id": "r2", "score": "N/A"},  # This will cause a ValueError in parse_score
]

try:
    results = process_all(bad_data)
except ValueError:
    print("Full traceback (read from bottom to top):")
    print()
    traceback.print_exc()
    print()
    print("Diagnosis:")
    print("  The error is in parse_score() — but the BAD DATA came from record r2.")
    print("  The fix is in process_record() or wherever records are validated.")

---
## 4. Common Bug Patterns

These patterns appear constantly in research code. Learn to recognize them.

In [ ]:
# Pattern 1: Off-by-one errors
# Python ranges are exclusive at the end: range(n) gives 0, 1, ..., n-1

items = ["a", "b", "c", "d", "e"]

# Bug: only processes first 4, misses last item
print("Buggy (misses last item):")
for i in range(len(items) - 1):  # Should be range(len(items))
    print(f"  items[{i}] = {items[i]}")

print("\nFixed:")
for i in range(len(items)):
    print(f"  items[{i}] = {items[i]}")

# Or better yet — just iterate directly:
print("\nPythonic (no index needed):")
for item in items:
    print(f"  {item}")

In [ ]:
# Pattern 2: Mutating a list while iterating over it
# This causes items to be skipped silently — a nasty bug

scores = [0.5, 0.9, 0.3, 0.2, 0.8]

# Bug: removing while iterating skips elements
buggy = scores.copy()
for score in buggy:
    if score < 0.5:
        buggy.remove(score)  # DON'T do this!
print(f"Buggy result (0.3 was skipped): {buggy}")  # 0.2 may survive

# Fix 1: build a new list with a list comprehension
filtered = [s for s in scores if s >= 0.5]
print(f"Fixed with comprehension: {filtered}")

# Fix 2: iterate over a copy
to_filter = scores.copy()
for score in scores[:]:
    if score < 0.5:
        to_filter.remove(score)
print(f"Fixed by iterating copy: {to_filter}")

---
## 5. A Buggy Function — Worked Example

Let's walk through debugging a function step by step.

In [ ]:
# Bug 1: SyntaxError — assignment (=) instead of comparison (==)
# Python will catch this before the code even runs.

# The following code has a SyntaxError — it's shown as a comment so the
# notebook doesn't halt. In a real session you'd see this error immediately.

# def compute_accuracy(results):
#     correct = 0
#     for r in results:
#         if r["correct"] = True:   # SyntaxError! This is assignment, not comparison
#             correct += 1
#     return correct / len(results)

# Error message:
# SyntaxError: invalid syntax
#   if r["correct"] = True:
#                   ^
# Python is telling you: you can't use = inside an if condition.
# The fix: use == for comparison.

def compute_accuracy(results):
    """Compute the fraction of correct results. FIXED version."""
    correct = 0
    for r in results:
        if r["correct"] == True:  # Fixed: == for comparison
            correct += 1
    return correct / len(results)

# Even better Python style:
def compute_accuracy_v2(results):
    correct = sum(1 for r in results if r["correct"])  # Truthy check
    return correct / len(results)

test = [{"correct": True}, {"correct": False}, {"correct": True}]
print(f"Accuracy: {compute_accuracy(test):.2%}")
print(f"Accuracy v2: {compute_accuracy_v2(test):.2%}")

In [ ]:
# Bug 2: Missing key — adding defensive checks

def get_flagged(outputs):
    """Return outputs where flagged=True."""
    flagged = []
    for output in outputs:
        if output["flagged"]:  # KeyError if 'flagged' key is missing!
            flagged.append(output)
    return flagged

# This works fine with clean data:
good_data = [
    {"id": 1, "text": "safe response", "flagged": False},
    {"id": 2, "text": "bad response", "flagged": True},
]
print(f"Works with clean data: {get_flagged(good_data)}")

# But fails with messy real-world data:
messy_data = [
    {"id": 1, "text": "safe", "flagged": False},
    {"id": 2, "text": "old record"},  # Missing 'flagged' key!
]
try:
    get_flagged(messy_data)
except KeyError as e:
    print(f"KeyError on messy data: {e}")

# Defensive version:
def get_flagged_safe(outputs):
    """Return outputs where flagged=True. Handles missing 'flagged' key."""
    flagged = []
    for output in outputs:
        if output.get("flagged", False):  # .get() with default False
            flagged.append(output)
    return flagged

print(f"Works with messy data: {get_flagged_safe(messy_data)}")

---
## 6. Intro to `pdb` — Python's Built-in Debugger

Sometimes print debugging isn't enough and you want to pause execution and inspect state interactively.
Python has a built-in debugger: `pdb`.

**You don't need to master `pdb` yet** — but you should know it exists.

In [ ]:
# Two ways to drop into the debugger:

# Method 1: breakpoint() — modern Python 3.7+
# Just add this line where you want to pause:
#
#   breakpoint()
#
# In a terminal this opens an interactive prompt where you can:
#   n  -> run the next line
#   s  -> step into a function call
#   p x -> print the value of x
#   l  -> show surrounding code
#   q  -> quit the debugger
#   c  -> continue running

# Method 2: import pdb; pdb.set_trace() — older style, same effect
#   import pdb; pdb.set_trace()

# In Jupyter notebooks, breakpoint() may behave differently.
# The most useful Jupyter-native alternative is to use %debug after an error.

# For now: print debugging + assert statements will handle 95% of your cases.
# Learn pdb when you need to debug complex interactive programs or scripts.

print("pdb is available — try it in a terminal script when you're ready.")
print("For now, master print debugging and assert statements first.")

---
## Exercises

### Exercise 1: Debug an Averaging Function

The function below is supposed to return the average of a list of numbers,
but it returns the wrong result. Use print statements to find the bug and fix it.

In [ ]:
# Buggy function
def compute_average(numbers):
    total = 0
    for n in numbers:
        total += n
    return total / (len(numbers) - 1)  # Bug is here!

# Test with known values — average of [2, 4, 6] should be 4.0

result = compute_average([2, 4, 6])
try:
    assert result == 4.0
except AssertionError as e:
    print(f"Got: {result}, Expected: 4.0")

In [ ]:
# YOUR FIX HERE
# Add print statements to trace the bug, then fix it

def compute_average_fixed(numbers):
    total = 0
    for n in numbers:
        total += n
    return total / len(numbers)  # Fix this line

result = compute_average_fixed([2, 4, 6])
print(f"Result: {result}")

In [ ]:
check_equal(compute_average_fixed([2, 4, 6]), 4.0, "Average of [2,4,6] should be 4.0")
check_equal(compute_average_fixed([10, 20, 30, 40]), 25.0, "Average of [10,20,30,40] should be 25.0")

### Exercise 2: Add Assert Statements

The function below works correctly, but has no input validation.
Add `assert` statements to validate inputs before the computation runs.

In [ ]:
# Starting point — no validation
def compute_weighted_average(scores, weights):
    """Compute a weighted average. Weights should sum to 1.0."""
    # TODO: Add assert statements to check:
    # 1. scores is a list
    # 2. weights is a list
    # 3. len(scores) == len(weights)
    # 4. len(scores) > 0
    # 5. All weights are between 0 and 1
    # 6. weights sum to approximately 1.0 (allow small floating point error)
    
    return sum(s * w for s, w in zip(scores, weights))

# Should work:
result = compute_weighted_average([0.8, 0.6, 0.9], [0.5, 0.3, 0.2])

print(f"Weighted average: {result:.4f}")

In [ ]:
# YOUR FIX HERE — add assert statements

def compute_weighted_average_safe(scores, weights):
    """Compute a weighted average. Weights should sum to 1.0."""
    # Add your assert statements here:

    return sum(s * w for s, w in zip(scores, weights))

# Test with valid inputs
result = compute_weighted_average_safe([0.8, 0.6, 0.9], [0.5, 0.3, 0.2])
print(f"Result: {result:.4f}")

# Test that it catches bad inputs
bad_caught = False
try:
    compute_weighted_average_safe([0.8, 0.6], [0.5, 0.3, 0.2])  # Length mismatch!
except AssertionError:
    bad_caught = True
    print("Correctly caught length mismatch")

print(f"Bad input was caught: {bad_caught}")

In [ ]:
check_equal(round(compute_weighted_average_safe([0.8, 0.6, 0.9], [0.5, 0.3, 0.2]), 4),
            0.76, "Weighted average should be 0.76")
check_equal(bad_caught, True, "Should catch length mismatch with AssertionError")

### Exercise 3: Fix a Silent Logic Error

The function below doesn't crash, but it returns wrong results.
Use print statements to trace the logic and find the bug.

In [ ]:
# Buggy function: should return only results where score > threshold
def filter_high_scores(results, threshold=0.7):
    """Return only results where score > threshold."""
    high = []
    for r in results:
        if r["score"] < threshold:  # Bug: should be >
            high.append(r)
    return high

test_results = [
    {"id": 1, "score": 0.9},
    {"id": 2, "score": 0.5},
    {"id": 3, "score": 0.8},
    {"id": 4, "score": 0.3},
]

# Should return records 1 and 3 (scores 0.9 and 0.8)
output = filter_high_scores(test_results, threshold=0.7)

print(f"High scorers: {[r['id'] for r in output]}")
print(f"Expected: [1, 3], Got: {[r['id'] for r in output]}")

In [ ]:
# YOUR FIX HERE

def filter_high_scores_fixed(results, threshold=0.7):
    """Return only results where score > threshold."""
    high = []
    for r in results:
        if r["score"] > threshold:  # Fix the condition
            high.append(r)
    return high

test_results = [
    {"id": 1, "score": 0.9},
    {"id": 2, "score": 0.5},
    {"id": 3, "score": 0.8},
    {"id": 4, "score": 0.3},
]

output = filter_high_scores_fixed(test_results, threshold=0.7)
result_ids = [r["id"] for r in output]
print(f"Result IDs: {result_ids}")

In [ ]:
check_equal(result_ids, [1, 3], "Should return IDs 1 and 3 (scores above 0.7)")

---
## Why This Matters

Debugging is a core skill. Research code that silently computes wrong numbers is dangerous —
you might make incorrect conclusions about model behavior.

Imagine running an evaluation of a model's safety properties and having a `<` instead of `>`
in your filter logic. You'd report that the model is unsafe when it isn't — or vice versa.
That kind of error could affect real decisions.

**Learning to add defensive checks and catch bugs early is essential AI safety engineering practice.**

The habits that protect you:
1. Test with small, known-answer examples before running on full data
2. Add `assert` statements at the boundaries of functions
3. Print intermediate values when something looks wrong
4. Always ask: "what did I expect vs. what actually happened?"

---
## Summary

- **Print debugging**: label your prints, print types and lengths, add separators for readability
- **assert statements**: document assumptions, fail loudly at the point of the bug
- **Tracebacks**: read bottom-to-top, the error type and message are the last line
- **Common patterns**: off-by-one in ranges, mutation during iteration, `=` vs `==`
- **pdb / breakpoint()**: available for interactive debugging in scripts
- **Mindset**: what did I expect vs. what actually happened?

**Next:** [03 — Review: Fix Buggy Analysis Code](03_review_and_practice.ipynb)